In [6]:
# READ THREE CSVS FROM 7.1 AND 7.2 AND CREATE Project_List

import pandas as pd
import os 

# === CONFIG ===
INPUT_DIR = r"C:\GitHub\Android-Mobile-Apps"
OUTPUT_PATH = os.path.join(INPUT_DIR, "7.3-Project_List.csv")

# === LOAD FILES ===
yml_df = pd.read_csv(os.path.join(INPUT_DIR, "YML_List.csv"))
api_summary_df = pd.read_csv(os.path.join(INPUT_DIR, "7.2-Project_List_API.csv"))
api_details_df = pd.read_csv(os.path.join(INPUT_DIR, "7.2-Project_List_API_Details.csv"))

# === Define GitHub Action based on test_type ===
def detect_github_action_from_test_type(test_type_series):
    def has_github_type(test_type_str):
        if pd.isna(test_type_str):
            return False
        test_types = [t.strip().lower() for t in str(test_type_str).split(',')]
        return any(t.startswith("github") for t in test_types)
    return test_type_series.apply(has_github_type)

yml_df['GitHub Action'] = detect_github_action_from_test_type(yml_df['test_type'])

# === Define Third_Party and Third_Party Test based on test_type only ===
def detect_third_party(test_type_str):
    if pd.isna(test_type_str):
        return False
    test_types = [t.strip().lower() for t in str(test_type_str).split(',')]
    return any(t not in {'none', 'other'} and not t.startswith('github') for t in test_types)

def extract_third_party_tests(test_type_str):
    if pd.isna(test_type_str):
        return ''
    test_types = [t.strip() for t in str(test_type_str).split(',')]
    return ', '.join(sorted(set(
        t for t in test_types if t and not t.lower().startswith('github') and t.lower() not in {'none', 'other'}
    )))

yml_df['Third_Party'] = yml_df['test_type'].apply(detect_third_party)
yml_df['Third_Party Test'] = yml_df['test_type'].apply(extract_third_party_tests)

# === GitHub Action Test Aggregation ===
github_tests = yml_df[yml_df['GitHub Action']]
github_test_types = github_tests.groupby('project')['test_type'].apply(
    lambda x: ', '.join(sorted({
        str(t).strip() for t in ', '.join(x.dropna().astype(str)).split(',')
        if t and str(t).strip().lower() not in {'none', 'nan', 'other'}
    }))
).reset_index(name='GitHub Action Test')

# === Create API presence matrix ===
api_details_df['API_COL'] = 'API_' + api_details_df['api_level'].astype(str)
api_presence = pd.crosstab(api_details_df['project'], api_details_df['API_COL']).astype(bool)
api_presence['Total API'] = api_presence.sum(axis=1)

# === CI Platform Aggregation ===
ci_platforms_all = yml_df.groupby('project')['ci_platform']\
    .apply(lambda x: ', '.join(sorted(set(x)))).reset_index(name='CI Platform')

ci_platforms_instr_only = yml_df[yml_df['instrumentation_test']].groupby('project')['ci_platform']\
    .apply(lambda x: ', '.join(sorted(set(x)))).reset_index(name='CI Platform_Modified')

# === Project-level summary ===
yml_summary = yml_df.groupby('project').agg({
    'unit_test': 'any',
    'instrumentation_test': 'any',
    'GitHub Action': 'any',
    'Third_Party': 'any',
    'Third_Party Test': lambda x: ', '.join(sorted(set(x) - {''})) if any(x) else ''
}).reset_index().rename(columns={
    'unit_test': 'Unit Test',
    'instrumentation_test': 'Instrumentation Testing'
})

# === Define Test Type Grouping ===
def classify_test_type_group(test_type_str):
    if pd.isna(test_type_str):
        return 'None'
    
    test_types = [t.strip().lower() for t in str(test_type_str).split(',') if t.strip()]
    
    github_types = {'github_emulator_full', 'github_emulator_compact', 'github_emulator_manual', 'github_gmd'}
    firebase_types = {'firebase_full', 'firebase_compact'}
    browserstack_types = {'browserstack'}
    manual_3rdparty_types = {'gitlab ci_emulator_manual', 'circleci_emulator_manual', 'travis ci_emulator_manual'}
    
    has_github = any(t in github_types for t in test_types)
    has_firebase = any(t in firebase_types for t in test_types)
    has_browserstack = any(t in browserstack_types for t in test_types)
    has_manual = any(t in manual_3rdparty_types for t in test_types)

    if has_firebase or has_browserstack:
        return 'ThirdParty_Service'
    elif has_manual:
        return 'Manual_ThirdParty'
    elif has_github:
        return 'GitHub_Hosted'
    else:
        return 'None'

# Apply grouping
yml_df['Test_Type_Grouped'] = yml_df['test_type'].apply(classify_test_type_group)

# === Aggregate the grouped type per project ===
grouped_test_types = yml_df.groupby('project')['Test_Type_Grouped'].apply(
    lambda x: ', '.join(sorted(set(x)))
).reset_index(name='Test_Type_Grouped')



# === Merge all together ===
merged = yml_summary.merge(api_summary_df[['project']], on='project', how='outer')
merged = merged.merge(api_presence, on='project', how='left').fillna(False)
merged = merged.merge(api_summary_df[['project', 'yml_count', 'yaml_errors']], on='project', how='left').fillna(0)
merged = merged.merge(ci_platforms_all, on='project', how='left')
merged = merged.merge(ci_platforms_instr_only, on='project', how='left')
merged = merged.merge(github_test_types, on='project', how='left')
merged = merged.merge(grouped_test_types, on='project', how='left')


# === Final export ===
output_cols = [
    'project', 'Unit Test', 'Instrumentation Testing', 'GitHub Action', 'GitHub Action Test',
    'Third_Party', 'Third_Party Test', 'Test_Type_Grouped',
    'CI Platform', 'CI Platform_Modified'
] + sorted([col for col in api_presence.columns if col.startswith("API_")]) + ['Total API']


final_df = merged[output_cols + ['yml_count', 'yaml_errors']]
final_df.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Final project list saved to: {OUTPUT_PATH}")


✅ Final project list saved to: C:\GitHub\Android-Mobile-Apps\7.3-Project_List.csv


C:\Users\Admin\AppData\Local\Temp\ipykernel_22076\3828977940.py:115: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged = merged.merge(api_presence, on='project', how='left').fillna(False)
